# human_eval_build
Freezes the pair list for the blind head-to-head. 150 scored pairs, 50 per contrast, each contrast being the 512 baseline against one finalist on the 488-row domain test split. 15 of the 150 are repeated later in the sequence with the sides swapped, giving 165 comparisons and a self-agreement estimate.

Sampling is stratified on the GPT-4.1 per-row margin so the agreement-against-margin curve has points in every band, and on `ref_kind` within each band so each margin band is sampled across the available levels, which reduces rather than eliminates confounding between provenance and margin. Both are properties of the row, fixed before rating.

Side assignment is randomised per pair and written to a separate file the rater interface never opens. Write-once: the analysis keys on this list and on the rater's responses to it.

CPU runtime, no compute units, no API calls.

## 1. Scores, split, arms

In [ ]:
from google.colab import drive
import os, json, math, random
from collections import defaultdict, Counter
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
SC    = f"{DRIVE}/scores"
HE    = f"{DRIVE}/human_eval"
os.makedirs(HE, exist_ok=True)

SEED    = 20260815
N_PER   = 50            # scored pairs per contrast
N_REPEAT= 15            # of the 150, re-shown later with the sides swapped
QUOTA   = {"tie": 12, "0-0.5": 12, "0.5-1.5": 13, ">1.5": 13}

# The baseline test images predate the gen/{DATASET}/{RUN} layout: omnigen2_generate wrote them
# to gen/baseline as test_{i:05d}.png, and score_baseline, judge_sd and baseline_embed all read
# them from there. The arms follow the later layout. Paths are stored per row, relative to DRIVE.
BASE = ("baseline", "test__baseline", "gen/baseline/test_{i:05d}.png")   # label, scores, path template
ARMS = [("lora_r8_step1000",     "test__lora_r8_step1000",     "gen/test/lora_r8_step1000/{i:05d}.png"),
        ("qlora_r8_step2000",    "test__qlora_r8_step2000",     "gen/test/qlora_r8__step2000/{i:05d}.png"),
        ("sp_iii_p139_step2000", "test__sp_iii_p139_step2000", "gen/test/sp_iii_p139__step2000/{i:05d}.png")]

rows = [json.loads(l) for l in open(f"{DRIVE}/test.jsonl", encoding="utf-8")]
uid  = lambda r: r["output_image"] + "|" + "|".join(sorted(r["input_images"]))
ROW  = {uid(r): (i, r) for i, r in enumerate(rows)}

def load(name):
    d = {}
    for l in open(f"{SC}/{name}.jsonl", encoding="utf-8"):
        r = json.loads(l)
        if "error" not in r: d[r["uid"]] = r
    return d

ov = lambda d: math.sqrt(max(d["PF"], 0) * max(d["SC"], 0))
B  = load(BASE[1])
print(f"test split {len(rows)} rows | baseline scored {len(B)}")
for lab, f, _ in ARMS:
    print(f"  {lab:24s} scored {len(load(f))}")

Mounted at /content/drive
test split 488 rows | baseline scored 488
  lora_r8_step1000         scored 488
  qlora_r8_step2000        scored 488
  sp_iii_p139_step2000     scored 488


## 2. Sample
Bands are the absolute GPT-4.1 Overall margin on the row. A rater who agrees with the judge on wide margins and sits at chance on narrow ones is the expected shape; sampling only from the middle would hide both ends.

In [ ]:
def band(x):
    a = abs(x)
    return "tie" if a == 0 else ("0-0.5" if a <= 0.5 else ("0.5-1.5" if a <= 1.5 else ">1.5"))

rnd = random.Random(SEED)
pairs = []
for lab, f, gendir in ARMS:
    A = load(f)
    K = sorted(set(A) & set(B))
    byband = defaultdict(list)
    for k in K:
        byband[band(ov(A[k]) - ov(B[k]))].append(k)
    picked = []
    for b, q in QUOTA.items():
        pool = byband[b]
        # spread the quota over ref_kind so each band covers the available levels; this reduces
        # rather than eliminates confounding between provenance and margin
        bykind = defaultdict(list)
        for k in pool: bykind[B[k]["ref_kind"]].append(k)
        for v in bykind.values(): rnd.shuffle(v)
        kinds, take = sorted(bykind), []
        while len(take) < q:
            progressed = False
            for kd in kinds:
                if bykind[kd] and len(take) < q:
                    take.append(bykind[kd].pop()); progressed = True
            if not progressed: break
        assert len(take) == q, f"{lab} {b}: only {len(take)} of {q} available"
        picked += take
    assert len(picked) == N_PER and len(set(picked)) == N_PER, (lab, len(picked))
    for k in picked:
        i, r = ROW[k]
        pairs.append({"contrast": lab, "uid": k, "row": i,
                      "instruction": r["instruction"],
                      "input_images": r["input_images"],
                      "ref_kind": B[k]["ref_kind"], "n_refs": B[k]["n_refs"],
                      "gpt_base": round(ov(B[k]), 4), "gpt_arm": round(ov(A[k]), 4),
                      "gpt_margin": round(ov(A[k]) - ov(B[k]), 4),
                      "gpt_band": band(ov(A[k]) - ov(B[k])),
                      "base_src": BASE[2].format(i=i), "arm_src": gendir.format(i=i)})

print(f"sampled {len(pairs)} pairs")
for lab, _, _ in ARMS:
    sub = [p for p in pairs if p["contrast"] == lab]
    print(f"  {lab:24s} n={len(sub)} | bands {dict(Counter(p['gpt_band'] for p in sub))}")
    print(f"  {'':24s} ref_kind {dict(Counter(p['ref_kind'] for p in sub))}")

sampled 150 pairs
  lora_r8_step1000         n=50 | bands {'tie': 12, '0-0.5': 12, '0.5-1.5': 13, '>1.5': 13}
                           ref_kind {'all_up': 18, 'mixed': 16, 'no_up': 16}
  qlora_r8_step2000        n=50 | bands {'tie': 12, '0-0.5': 12, '0.5-1.5': 13, '>1.5': 13}
                           ref_kind {'all_up': 18, 'mixed': 16, 'no_up': 16}
  sp_iii_p139_step2000     n=50 | bands {'tie': 12, '0-0.5': 12, '0.5-1.5': 13, '>1.5': 13}
                           ref_kind {'all_up': 18, 'mixed': 16, 'no_up': 16}


## 3. Order, sides, repeats
Contrasts are interleaved so a session is not a block of one arm. Repeats are drawn from the first half and re-inserted in the last third with the sides swapped, so a repeat cannot be recognised by its position on screen.

In [ ]:
rnd.shuffle(pairs)
for n, p in enumerate(pairs):
    p["tmp"] = n                                     # internal only, never written
    p["left_is"] = rnd.choice(["base", "arm"])       # kept out of the rater's file

half = len(pairs) // 2
repeats = []
for p in rnd.sample(pairs[:half], N_REPEAT):
    q = dict(p)
    q["repeat_of_tmp"] = p["tmp"]
    q["left_is"] = "arm" if p["left_is"] == "base" else "base"   # sides swapped
    repeats.append(q)

tail = pairs[int(len(pairs) * 2 / 3):]
slots = sorted(rnd.sample(range(len(tail) + len(repeats)), len(repeats)))
merged, it, ir = [], iter(tail), iter(repeats)
for n in range(len(tail) + len(repeats)):
    merged.append(next(ir) if n in slots else next(it))
ordered = pairs[:int(len(pairs) * 2 / 3)] + merged

# pair_id is assigned last, from the position in the final sequence. Deriving a repeat's id
# from its original would put the repeat marker, and the identity of the original, in the
# rater's file; a self-agreement estimate is only worth having if repeats are unrecognisable.
for n, p in enumerate(ordered):
    p["seq"] = n
    p["pair_id"] = f"p{n:04d}"
by_tmp = {p["tmp"]: p["pair_id"] for p in ordered if "repeat_of_tmp" not in p}
for p in ordered:
    if "repeat_of_tmp" in p:
        p["repeat_of"] = by_tmp[p["repeat_of_tmp"]]

gap = [n - next(m for m, q in enumerate(ordered) if q["pair_id"] == p["repeat_of"])
       for n, p in enumerate(ordered) if p.get("repeat_of")]
print(f"{len(ordered)} comparisons ({len(pairs)} scored + {len(repeats)} repeats)")
print(f"repeat separation: min {min(gap)}, median {sorted(gap)[len(gap)//2]}, max {max(gap)} comparisons")
print(f"left_is balance: {dict(Counter(p['left_is'] for p in ordered))}")
print(f"repeat ids are indistinguishable from the rest: "
      f"{sorted(p['pair_id'] for p in ordered if p.get('repeat_of'))[:4]} ...")
assert min(gap) >= 30, "a repeat sits too close to its original"
assert len({p["pair_id"] for p in ordered}) == len(ordered)

165 comparisons (150 scored + 15 repeats)
repeat separation: min 32, median 107, max 150 comparisons
left_is balance: {'base': 72, 'arm': 93}
repeat ids are indistinguishable from the rest: ['p0102', 'p0106', 'p0107', 'p0108'] ...


## 4. Write
Two files. `pairs.jsonl` is what the interface reads and carries no arm labels, no side assignment and no GPT scores. `key.jsonl` holds the mapping and is opened only by the analysis. Write-once.

In [ ]:
import shutil
PAIRS, KEY = f"{HE}/pairs.jsonl", f"{HE}/key.jsonl"
STAGE = f"{HE}/images"
if os.path.exists(PAIRS):
    raise SystemExit(f"exists, kept: {PAIRS}")
os.makedirs(STAGE, exist_ok=True)

# The generated images live in directories named after the arm. Referencing those paths from
# the rater's file would name the arm on every line, so each side is copied to a neutral file
# and only that name is exposed. 330 copies, one time.
BLIND = ["seq", "pair_id", "instruction", "input_images", "left_png", "right_png"]

# resolve every path first: a missing source partway through would leave the two jsonl files
# and the staging directory half written, and the write-once guard would then block a retry
for p in ordered:
    p["left_src"]  = p["base_src"] if p["left_is"] == "base" else p["arm_src"]
    p["right_src"] = p["arm_src"]  if p["left_is"] == "base" else p["base_src"]
    p["left_png"], p["right_png"] = f"{p['pair_id']}_A.png", f"{p['pair_id']}_B.png"
absent = [f"{DRIVE}/{s}" for p in ordered for s in (p["left_src"], p["right_src"])
          if not os.path.exists(f"{DRIVE}/{s}")]
assert not absent, f"{len(absent)} source images missing, first: {absent[:3]}"
print(f"all {len(ordered)*2} source images resolved")

with open(PAIRS, "w", encoding="utf-8") as fp, open(KEY, "w", encoding="utf-8") as fk:
    for p in ordered:
        for src, dst in ((p["left_src"], p["left_png"]), (p["right_src"], p["right_png"])):
            if not os.path.exists(f"{STAGE}/{dst}"):
                shutil.copyfile(f"{DRIVE}/{src}", f"{STAGE}/{dst}")
        fp.write(json.dumps({k: p[k] for k in BLIND}, ensure_ascii=False) + "\n")
        fk.write(json.dumps(p, ensure_ascii=False) + "\n")
print(f"staged {len(ordered)*2} images -> {STAGE}")
print(f"wrote {PAIRS} ({len(ordered)} rows, blind)")
print(f"wrote {KEY} ({len(ordered)} rows, side assignment, source paths and GPT margins)")

blind = [json.loads(l) for l in open(PAIRS, encoding="utf-8")]
fields = set().union(*[set(b) for b in blind])
leak = fields & {"contrast", "left_is", "gpt_margin", "gpt_band", "gpt_base", "gpt_arm",
                 "base_src", "arm_src", "repeat_of", "left_src", "right_src", "row", "uid", "tmp"}
assert not leak, f"the rater file leaks the fields {leak}"
# and nothing in the VALUES may name an arm or a directory
names = [a for _, _, a in ARMS] + [BASE[0], BASE[2], "baseline", "lora", "qlora", "sp_iii", "step", "_R"]
text = "\n".join(json.dumps(b[k]) for b in blind for k in ("pair_id", "left_png", "right_png"))
hit = sorted({n for n in names if n in text})
assert not hit, f"the rater file names {hit} in its values"
ids = [b["pair_id"] for b in blind]
assert all(len(i) == 5 and i[0] == "p" and i[1:].isdigit() for i in ids), "a pair_id is not opaque"
print(f"blind file fields: {sorted(fields)}")
print("no arm name, side, GPT score, row index or repeat marker reaches the rater,")
print("in a field name or in a value")

all 330 source images resolved
staged 330 images -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/human_eval/images
wrote /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/human_eval/pairs.jsonl (165 rows, blind)
wrote /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/human_eval/key.jsonl (165 rows, side assignment, source paths and GPT margins)
blind file fields: ['input_images', 'instruction', 'left_png', 'pair_id', 'right_png', 'seq']
no arm name, side, GPT score, row index or repeat marker reaches the rater,
in a field name or in a value


## 5. Images present

In [ ]:
missing = [p for p in ordered
           if not (os.path.exists(f"{STAGE}/{p['left_png']}") and os.path.exists(f"{STAGE}/{p['right_png']}"))]
print(f"staged images resolved: {len(ordered)-len(missing)}/{len(ordered)}")
assert not missing, [p["pair_id"] for p in missing[:5]]

import hashlib
def h(p): return hashlib.blake2b(open(p, "rb").read(), digest_size=8).hexdigest()
same = [p["pair_id"] for p in ordered if h(f"{STAGE}/{p['left_png']}") == h(f"{STAGE}/{p['right_png']}")]
print(f"pairs whose two sides are byte-identical: {len(same)}  (rate these tie)")
print(f"\nreference root expected by the interface: {DRIVE}/imgs (restored from imgs_subset.tar)")
print(f"staged image root: {STAGE}")

staged images resolved: 165/165
pairs whose two sides are byte-identical: 0  (rate these tie)

reference root expected by the interface: /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/imgs (restored from imgs_subset.tar)
staged image root: /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/human_eval/images
